# Triage Pilot — Phase 0 Environment Audit

Run top-to-bottom. Writes `env_audit_<timestamp>.json` to Drive at the path set in the config cell.

**Purpose:** Lock down the environment before any training work starts. Anything surprising here should be fixed now, not during Phase 1 debugging.

## 1. Config

In [ ]:
# Edit these for your setup
DRIVE_MOUNT_POINT = '/content/drive'
TRIAGE_ROOT = '/content/drive/MyDrive/setique/triage'  # will be created if missing
AUDIT_OUTPUT_DIR = f'{TRIAGE_ROOT}/phase0/env_audits'

# Pinned versions — matches requirements.txt (post-audit, 2026-04-23 Colab T4 runtime).
# bitsandbytes is intentionally omitted: not installed in the observed runtime and
# not required for Phase 0. Baseline/Phase 1 notebooks install it on demand.
PINNED = {
    'torch': '2.10.0',
    'transformers': '5.0.0',
    'datasets': '4.0.0',
    'tokenizers': '0.22.2',
    'accelerate': '1.13.0',
}

## 2. Mount Drive and prepare directories

In [ ]:
import os
from google.colab import drive

drive.mount(DRIVE_MOUNT_POINT, force_remount=False)
os.makedirs(TRIAGE_ROOT, exist_ok=True)
os.makedirs(AUDIT_OUTPUT_DIR, exist_ok=True)
print(f'Triage root ready: {TRIAGE_ROOT}')
print(f'Audit output dir: {AUDIT_OUTPUT_DIR}')

## 3. GPU + CUDA check

In [ ]:
import subprocess

nvidia_smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(nvidia_smi)

# Extract GPU name for the audit record
gpu_name = 'unknown'
for line in nvidia_smi.splitlines():
    if 'Tesla' in line or 'A100' in line or 'L4' in line or 'T4' in line or 'V100' in line:
        gpu_name = line.strip()
        break

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
cuda_version = torch.version.cuda if cuda_available else None
device_name = torch.cuda.get_device_name(0) if cuda_available else None
device_capability = torch.cuda.get_device_capability(0) if cuda_available else None
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if cuda_available else None

# BF16 support: compute capability 8.0+ = Ampere or newer (A100, L4, H100). T4 is 7.5 = no BF16.
bf16_supported = cuda_available and device_capability is not None and device_capability[0] >= 8

print(f'CUDA available: {cuda_available}')
print(f'CUDA version:   {cuda_version}')
print(f'Device:         {device_name}')
print(f'Capability:     {device_capability}')
print(f'VRAM:           {vram_total_gb:.2f} GB' if vram_total_gb else 'VRAM: n/a')
print(f'BF16 native:    {bf16_supported}')

## 4. Library versions

In [ ]:
import importlib
import importlib.metadata as md

def get_version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

installed = {pkg: get_version(pkg) for pkg in PINNED.keys()}
mismatches = {pkg: (PINNED[pkg], installed[pkg]) for pkg in PINNED if installed[pkg] != PINNED[pkg]}

for pkg, ver in installed.items():
    status = 'OK' if ver == PINNED[pkg] else f'MISMATCH (want {PINNED[pkg]})'
    print(f'{pkg:15s} {str(ver):15s} {status}')

# Note: Colab often has newer versions than we'd pin. Don't auto-upgrade — record and decide.
if mismatches:
    print('\nMismatches detected. Decide whether to pin-up or pin-down before Phase 1.')

## 5. FP16 mixed precision smoke test

In [ ]:
import torch
import torch.nn as nn

fp16_ok = False
fp16_err = None
if cuda_available:
    try:
        model = nn.Linear(256, 256).cuda()
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        scaler = torch.amp.GradScaler('cuda')
        x = torch.randn(32, 256, device='cuda')
        y = torch.randn(32, 256, device='cuda')
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            out = model(x)
            loss = ((out - y) ** 2).mean()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        fp16_ok = True
        print(f'FP16 mixed precision: OK (loss={loss.item():.4f})')
    except Exception as e:
        fp16_err = repr(e)
        print(f'FP16 smoke test FAILED: {fp16_err}')
else:
    print('No CUDA — FP16 test skipped')

## 6. bitsandbytes import check

In [ ]:
bnb_ok = False
bnb_installed = False
bnb_err = None
try:
    import bitsandbytes as bnb
    bnb_installed = True
    bnb_version = bnb.__version__
    # Minimal INT8 layer construction — doesn't need a forward pass to validate the import is functional
    _ = bnb.nn.Linear8bitLt(16, 16, has_fp16_weights=False)
    bnb_ok = True
    print(f'bitsandbytes: OK (version {bnb_version})')
except ModuleNotFoundError as e:
    bnb_err = repr(e)
    print('bitsandbytes: not installed, not required at this phase')
    print('  (baseline/Phase 1 notebooks install it on demand; safe to proceed)')
except Exception as e:
    bnb_err = repr(e)
    print(f'bitsandbytes: installed but smoke test FAILED: {bnb_err}')

## 7. Determinism test

Same seed, same input → same loss, twice. If this fails, something in the stack is nondeterministic and experiments won't be reproducible.

In [ ]:
import random
import numpy as np

def run_with_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if cuda_available:
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(False)  # keep False for speed; flip to True if needed
    model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
    if cuda_available:
        model = model.cuda()
    x = torch.randn(16, 64, device='cuda' if cuda_available else 'cpu')
    y = torch.randn(16, 1, device='cuda' if cuda_available else 'cpu')
    out = model(x)
    return ((out - y) ** 2).mean().item()

loss_a = run_with_seed(42)
loss_b = run_with_seed(42)
determinism_ok = abs(loss_a - loss_b) < 1e-6
print(f'Loss run 1: {loss_a}')
print(f'Loss run 2: {loss_b}')
print(f'Deterministic: {determinism_ok}')

## 8. Drive space check

In [ ]:
statvfs = os.statvfs(TRIAGE_ROOT)
drive_free_gb = (statvfs.f_bavail * statvfs.f_frsize) / (1024**3)
drive_total_gb = (statvfs.f_blocks * statvfs.f_frsize) / (1024**3)
drive_ok = drive_free_gb >= 20
print(f'Drive total: {drive_total_gb:.1f} GB')
print(f'Drive free:  {drive_free_gb:.1f} GB')
print(f'Headroom OK (>=20GB): {drive_ok}')

## 9. Write audit report

In [ ]:
import json
from datetime import datetime

audit = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'environment': 'colab',
    'gpu': {
        'name': device_name,
        'capability': list(device_capability) if device_capability else None,
        'vram_gb': round(vram_total_gb, 2) if vram_total_gb else None,
        'cuda_version': cuda_version,
        'bf16_native': bf16_supported,
        'nvidia_smi_raw': nvidia_smi,
    },
    'libraries': {
        'pinned': PINNED,
        'installed': installed,
        'mismatches': mismatches,
    },
    'smoke_tests': {
        'fp16_ok': fp16_ok,
        'fp16_error': fp16_err,
        'bitsandbytes_ok': bnb_ok,
        'bitsandbytes_error': bnb_err,
        'determinism_ok': determinism_ok,
        'determinism_loss_run1': loss_a,
        'determinism_loss_run2': loss_b,
    },
    'drive': {
        'total_gb': round(drive_total_gb, 1),
        'free_gb': round(drive_free_gb, 1),
        'headroom_ok': drive_ok,
    },
    'notes': [
        'T4 has no BF16; use FP16 mixed precision throughout Phase 0/1.',
        'Forge (RTX 2070) is reserved for PAI/E2B work — not this pilot.',
        'If FP16 smoke test failed, check PyTorch/CUDA version compatibility.',
    ],
}

stamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
out_path = f'{AUDIT_OUTPUT_DIR}/env_audit_{stamp}.json'
with open(out_path, 'w') as f:
    json.dump(audit, f, indent=2)

print(f'Audit written to: {out_path}')
print(json.dumps(audit, indent=2))

## 10. Checkpoint 1 verdict

Review the output above. All of these should be true before marking Checkpoint 1 green:

- GPU detected and VRAM matches expectations for the instance type
- FP16 smoke test passed
- Determinism test passed
- Drive free space OK
- Library mismatches are either zero or consciously accepted

bitsandbytes is **not required** for Checkpoint 1. If it's not installed, that's acceptable — the baseline/Phase 1 notebooks install it on demand. If it IS installed but the smoke test failed, that's a red flag to investigate.

If anything else is red, fix it before moving to Checkpoint 2.